# The Bigger Picture

To improve and customize our learning experience, we are building a tools that extracts clean textual transcripts from Microsoft Learn lessons.

**The pipeline will look like the following:**
- URL
- HTML download
- Content extraction
- Clean transcript
- Saved text files

**Folders:**
- downloaded_HTMLs/
- clean_transcripts/

## Install Dependencies and Import Libraries

This project relies on two external libraries commonly used for web scraping:

- *requests* → downloads webpages
- *BeautifulSoup* → parses HTML and allows navigation through page elements

If they are not installed yet, the following cell can install them.

Additionally, we import the Python modules used throughout the notebook:

- `os` handles file paths and directory management
- `requests` retrieves webpages
- `BeautifulSoup` parses HTML structures
- `urllib.parse` helps extract filenames from URLs
- `re` allows pattern matching for filtering unwanted text

In [20]:
#!pip install requests beautifulsoup4

In [18]:
import os
import requests
import re
from bs4 import BeautifulSoup
from urllib.parse import urlparse


## Folder Setup

This cell creates the directories used to store intermediate and final outputs.

Two folders are created if they do not already exist:
- `downloaded_HTMLs/` for raw webpages
- `clean_transcripts/` for processed transcripts

Separating these outputs makes the workflow easier to debug.

In [3]:
HTML_DIR = "downloaded_HTMLs"
TXT_DIR = "clean_transcripts"

os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(TXT_DIR, exist_ok=True)

print("Folders ready.")

Folders ready.


## Define Lessons URLs

This cell defines the list of Microsoft Learn lesson pages to process.

Each URL corresponds to one lesson. The pipeline will iterate through this list and process each page automatically.

In [23]:
urls = [
"https://learn.microsoft.com/en-us/training/modules/describe-cloud-compute/4-describe-shared-responsibility-model",
"https://learn.microsoft.com/en-us/training/modules/describe-cloud-compute/6-describe-consumption-based-model"
]

## Filename Sanitization

Sometimes URLs make terrible filenames, containing characters that are not suitable for filenames.

This helper function converts a URL into a safe filename by:
- removing the domain
- replacing / with _
- preserving the lesson structure

In [24]:
def sanitize_filename(url):

    parsed = urlparse(url)
    name = parsed.path.strip("/").replace("/", "_")

    if not name:
        name = "index"

    return name

In [25]:
# Test
sanitize_filename(urls[0])

'en-us_training_modules_describe-cloud-compute_4-describe-shared-responsibility-model'

### HTML Downloader

Here we download each URL page as a HTML file.

The process:
1. Send an HTTP request to the webpage
2. Retrieve the HTML source
3. Save the page locally in the `downloaded_HTMLs` folder

The User-Agent header is included to prevent request blocking by the website.

In [ ]:
def download_html(url):

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    response.raise_for_status()

    response.encoding = response.apparent_encoding

    filename = sanitize_filename(url) + ".html"
    path = os.path.join(HTML_DIR, filename)

    with open(path, "w", encoding="utf-8") as f:
        f.write(response.text)

    return path

## Inspect Page Structure

This step is looking inside the HTML page to understand the structure before scraping it. Microsoft Learn usually places the lesson text inside `<main>`.

Essentially:
- Open the downloaded HTML file.
- Give it to BeautifulSoup, which converts the raw HTML into a tree structure you can search.
- Checks if BeautifulSoup is parsing correctly and shows the page title tag.

In [ ]:
html_path = download_html(urls[0])

with open(html_path, encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

print(soup.title)
# Is there a <main> section in this page?
soup.find("main")

<title>Describe the shared responsibility model - Training | Microsoft Learn</title>


### Extract Main Article Content

In [30]:
def get_content_root(soup):

    main = soup.find("main")

    if not main:
        return soup

    article = main.find("article")

    if article:
        return article

    return main

In [31]:
# Test
content_root = get_content_root(soup)
content_root.name

'main'

### Extract Structured Text

In [ ]:
def extract_clean_text(html_path):

    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    root = get_content_root(soup)

    content = []

    for tag in root.find_all(["h1","h2","h3","h4","p","li"]):

        text = tag.get_text(" ", strip=True)

        if not text:
            continue

        # This removes reading time lines like "3 minutes"
        if re.match(r"^\d+\s+minute", text.lower()):
            continue

        if tag.name.startswith("h"):
            content.append("\n" + text.upper())
            content.append("-"*len(text))

        elif tag.name == "li":
            content.append(f"- {text}")

        else:
            content.append(text)

    # remove duplicates
    content = list(dict.fromkeys(content))

    return "\n".join(content)

In [32]:
# Test
text = extract_clean_text(html_path)
print(text[:2000])


DESCRIBE THE SHARED RESPONSIBILITY MODEL
----------------------------------------
You may have heard of the shared responsibility model, but you may not understand what it means or how it impacts cloud computing.
Start with a traditional corporate datacenter. The company is responsible for maintaining the physical space, ensuring security, and maintaining or replacing the servers if anything happens. The IT department is responsible for maintaining all the infrastructure and software needed to keep the datacenter up and running. They’re also likely to be responsible for keeping all systems patched and on the correct version.
With the shared responsibility model, these responsibilities get shared between the cloud provider and the consumer. Physical security, power, cooling, and network connectivity are the responsibility of the cloud provider. The consumer isn’t collocated with the datacenter, so it wouldn’t make sense for the consumer to have any of those responsibilities.
At the sam

### Save Transcript

In [ ]:
gh repo create microsoft-learn-transcriber --private --source=. --remote=origin --push
